# try3 inference: designated submission (0.88078 public, 0.87910 local)

Writes Kaggle submission 58102347 from its saved final layer and checks it against the submitted CSV. Nothing is trained here.

The final layer is `p = sigmoid(parent_logit + features @ weights)`:

- `parent_logit` is the output of the 0.879 detector ensemble (51 models plus corrections).
- `features` are 5 correction inputs, built from a 5-fold MLP trained with synthetic data.

Both are stored in `designated_088078/model/mixed_replay.npz`. The last section recomputes the MLP part from the raw test series.

To score new series with models you trained yourself (`python -m pipeline.train_gpu ...`), use `inference.ipynb` at the repo root.

Output: `id,break_probability`

In [ ]:
#settings, change these paths
DATA = 'data'                              #competition folder: test_series.parquet + sample_submission.csv
MODEL = 'try3/designated_088078/model'     #saved final layer + MLP folds
CSV = 'submission.csv'
OUT = 'try3_member'                        #scratch folder for the MLP check, gets created

In [ ]:
#imports
import os, sys, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
os.environ['CUDA_VISIBLE_DEVICES'] = ''   #cpu only

In [ ]:
#look at the new data
raw = pd.read_parquet(f'{DATA}/test_series.parquet')
print(raw.shape, raw['id'].nunique(), 'series')
raw.head()

In [ ]:
#plot a few, red line is the boundary we score
ids = raw['id'].unique()[:3]
fig, ax = plt.subplots(len(ids), 1, figsize=(10, 6))
for a, i in zip(ax, ids):
    s = raw[raw['id'] == i].sort_values('time')
    a.plot(s['value'].values, lw=.7)
    a.axvline((s['period'] == 0).sum(), ls='--', c='r')
    a.set_title(str(i))
plt.tight_layout()

In [ ]:
#the saved final layer
z = np.load(f'{MODEL}/mixed_replay.npz')
print('parent logits', z['parent_logits'].shape, ' correction features', z['features'].shape)
print('weights', np.round(z['weights'], 4))
shift = z['features'] @ z['weights']
print('logit shift from the correction: mean %.3f, max abs %.3f' % (shift.mean(), np.abs(shift).max()))
plt.figure(figsize=(6, 3))
plt.hist(expit(z['parent_logits']), 50, alpha=.6, label='parent ensemble')
plt.hist(expit(z['parent_logits'] + shift), 50, alpha=.6, label='with correction')
plt.legend(); plt.xlabel('break probability')

In [ ]:
#run it: applies the final layer, writes id,break_probability, checks it against the submitted csv
subprocess.run([sys.executable, 'try3/designated_088078/predict.py', '--data', DATA, '--out', CSV], check=True)

In [ ]:
#check the output
sub = pd.read_csv(CSV)
print(sub.shape)
print(sub.break_probability.describe())
sub.head()

## Optional: recompute the MLP part from raw series

Rebuilds the 361 MLP inputs from `test_series.parquet` and runs the 5 saved fold models on CPU (about 90 s with 2 workers). The result is the MLP's mean logit, not the full ensemble.

In [ ]:
subprocess.run([sys.executable, 'try3/designated_088078/member_cpu.py', '--data', DATA, '--model', MODEL,
                '--output', OUT, '--workers', '2'], check=True)
m = np.load(f'{OUT}/member_logits.npz')
print(m['test'].shape, 'mlp logits, mean %.3f' % m['test'].mean())